In [ ]:
import spacy
import json
import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
HOME_DIR = os.environ.get('HOME')
import phonlp
import warnings
from datasets import load_dataset
from tqdm import tqdm
import random
from pyvi import ViTokenizer

In [ ]:
print("Thư mục hiện tại:", os.getcwd())

In [ ]:
# 1. Initialize English NLP model (spaCy)
!python -m spacy download en_core_web_sm

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

nlp_en = spacy.load("en_core_web_sm")

save_dir = f'{HOME_DIR}/phonlp_weights'

if not os.path.exists(save_dir) or not os.listdir(save_dir):
    print("Đang tải model weights cho PhoNLP...")
    phonlp.download(save_dir=save_dir)

nlp_vi = phonlp.load(save_dir=save_dir)

# nlp_en = None
# nlp_vi = None

# def get_models():
#     global nlp_en, nlp_vi
#     if nlp_en is None:
#         nlp_en = spacy.load("en_core_web_sm")
#     if nlp_vi is None:
#         save_dir = './phonlp_weights'
#         if not os.path.exists(save_dir):
#             phonlp.download(save_dir=save_dir)
#         nlp_vi = phonlp.load(save_dir=save_dir)
#     return nlp_en, nlp_vi

# Mapping PhoNLP specific POS tags to Universal POS (UPOS) standard
PHONLP_TO_UPOS_MAP = {
    "N": "NOUN", "Np": "PROPN", "Nc": "NOUN", "Nb": "NOUN", "Nu": "NOUN", "Ny": "NOUN",
    "V": "VERB", "Vb": "VERB", "A": "ADJ", "P": "PRON", "L": "DET", "R": "ADV",
    "E": "ADP", "C": "CCONJ", "CH": "PUNCT", "M": "NUM", "T": "PART",
    "I": "INTJ", "X": "X", "Z": "X", "Y": "NOUN"
}

In [ ]:
def get_vietnamese_upos(vi_text, model_vi):
    segmented_text = ViTokenizer.tokenize(vi_text)
    annotation = model_vi.annotate(segmented_text)
    def flatten_to_strings(obj):
        if isinstance(obj, str):
            return [obj]
        elif isinstance(obj, (list, tuple)):
            result = []
            for item in obj:
                result.extend(flatten_to_strings(item))
            return result
        return []
    
    pos_tags_raw = flatten_to_strings(annotation[1])
    upos_list = [PHONLP_TO_UPOS_MAP.get(tag, "X") for tag in pos_tags_raw]
    return upos_list

In [ ]:
def process_translation_pair(en_text, vi_text, model_en, model_vi):
    doc_en = model_en(en_text)
    en_upos = [token.pos_ for token in doc_en]
    vi_upos = get_vietnamese_upos(vi_text, model_vi)

    # verb_count = en_upos.count("VERB")
    # sconj_count = en_upos.count("SCONJ")
    # cconj_count = en_upos.count("CCONJ")
    # en_length = len(en_upos) if len(en_upos) > 0 else 1
    # complexity_score = (verb_count * 1.0 + sconj_count * 3.0 + cconj_count * 1.5) / en_length

    # Calculate score for English
    en_length = len(en_upos) if len(en_upos) > 0 else 1
    en_hard = en_upos.count("SCONJ") * 3.0 + en_upos.count("AUX") * 2.0
    en_medium = en_upos.count("CCONJ") * 1.5 + en_upos.count("VERB") * 1.0
    en_score = (en_hard + en_medium) / en_length
    
    # Calculate score for Vietnamese (Substituting AUX with PART for tense/voice markers)
    vi_length = len(vi_upos) if len(vi_upos) > 0 else 1
    vi_hard = vi_upos.count("SCONJ") * 3.0 + vi_upos.count("PART") * 2.0
    vi_medium = vi_upos.count("CCONJ") * 1.5 + vi_upos.count("VERB") * 1.0
    vi_score = (vi_hard + vi_medium) / vi_length

    complexity_score = (en_score + vi_score) / 2.0
    
    # Calculate exact word (token) count based on NLP engine outputs
    # Add +1 for the [SPLIT] token in each block
    total_tokens = len(en_upos) + len(vi_upos) + 2
    en_token_count = len(en_upos)
    
    text_block = f"{en_text} \n\n {vi_text} [SPLIT]"
    pos_block = f"{' '.join(en_upos)} \n\n {' '.join(vi_upos)} [SPLIT]"
    
    return {
        "score": complexity_score,
        "word_count": total_tokens,
        "en_word_count": en_token_count,
        "en_text": en_text,
        "en_pos": " ".join(en_upos),
        "vi_text": vi_text,
		"vi_pos": " ".join(vi_upos),
        "text_block": text_block,
        "pos_block": pos_block
    }

In [ ]:
def sample_from_buckets(easy_b, med_b, hard_b, target_words, stage_id):
    '''
    Sample from the three buckets with a skewed curriculum distribution until we hit the target word count.'''
    sampled_items = []
    accumulated_words = 0
    
    while accumulated_words < target_words:
        if not easy_b and not med_b and not hard_b:
            break
            
        pool = []
        weights = []
        if easy_b:
            pool.append('easy')
            weights.append(0.60)
        if med_b:
            pool.append('medium')
            weights.append(0.30)
        if hard_b:
            pool.append('hard')
            weights.append(0.10)
            
        total_w = sum(weights)
        weights = [w / total_w for w in weights]
        
        chosen = random.choices(pool, weights=weights, k=1)[0]
        
        if chosen == 'easy':
            item = easy_b.pop()
        elif chosen == 'medium':
            item = med_b.pop()
        else:
            item = hard_b.pop()
            
        if accumulated_words + item["word_count"] > target_words:
            # Re-insert to avoid discarding data prematurely
            if chosen == 'easy': easy_b.append(item)
            elif chosen == 'medium': med_b.append(item)
            else: hard_b.append(item)
            break
            
        item["stage"] = stage_id
        sampled_items.append(item)
        accumulated_words += item["word_count"]
        
    return sampled_items, accumulated_words


In [ ]:
def calculate_dataset_scores(examples):
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    import spacy
    import phonlp
    from pyvi import ViTokenizer
    import os
    import torch
    import sys

    torch.set_num_threads(1)

    if not hasattr(sys, "nlp_en_cache"):
        sys.nlp_en_cache = spacy.load("en_core_web_sm")
        HOME_DIR = os.environ.get('HOME')
        save_dir = f'{HOME_DIR}/phonlp_weights'
        
        if not os.path.exists(save_dir):
            phonlp.download(save_dir=save_dir)
        sys.nlp_vi_cache = phonlp.load(save_dir=save_dir)

    model_en = sys.nlp_en_cache
    model_vi = sys.nlp_vi_cache

    PHONLP_TO_UPOS_MAP = {
        "N": "NOUN", "Np": "PROPN", "Nc": "NOUN", "Nb": "NOUN", "Nu": "NOUN", "Ny": "NOUN",
        "V": "VERB", "Vb": "VERB", "A": "ADJ", "P": "PRON", "L": "DET", "R": "ADV",
        "E": "ADP", "C": "CCONJ", "CH": "PUNCT", "M": "NUM", "T": "PART",
        "I": "INTJ", "X": "X", "Z": "X", "Y": "NOUN"
    }

    def process_pair(en_text, vi_text):
        if len(vi_text.split()) > 150 or len(en_text.split()) > 150:
            return {
                "score": -1.0,
                "word_count": 0,
                "en_word_count": 0,
                "en_pos": "",
                "vi_pos": ""
            }

        try:
            doc_en = model_en(en_text)
            en_upos = [token.pos_ for token in doc_en]
    
            segmented_text = ViTokenizer.tokenize(vi_text)
            annotation = model_vi.annotate(segmented_text)
    
            def flatten_to_strings(obj):
                if isinstance(obj, str):
                    return [obj]
                elif isinstance(obj, (list, tuple)):
                    result = []
                    for item in obj:
                        result.extend(flatten_to_strings(item))
                    return result
                return []
    
            pos_tags_raw = flatten_to_strings(annotation[1])
            vi_upos = [PHONLP_TO_UPOS_MAP.get(tag, "X") for tag in pos_tags_raw]
    
            en_length = max(len(en_upos), 1)
            en_hard = en_upos.count("SCONJ") * 3.0 + en_upos.count("AUX") * 2.0
            en_medium = en_upos.count("CCONJ") * 1.5 + en_upos.count("VERB") * 1.0
            en_score = (en_hard + en_medium) / en_length
    
            vi_length = max(len(vi_upos), 1)
            vi_hard = vi_upos.count("SCONJ") * 3.0 + vi_upos.count("PART") * 2.0
            vi_medium = vi_upos.count("CCONJ") * 1.5 + vi_upos.count("VERB") * 1.0
            vi_score = (vi_hard + vi_medium) / vi_length
    
            complexity_score = (en_score + vi_score) / 2.0
            total_tokens = len(en_upos) + len(vi_upos) + 2
            
            return {
                "score": complexity_score,
                "word_count": total_tokens,
                "en_word_count": len(en_upos),
                "en_pos": " ".join(en_upos),
                "vi_pos": " ".join(vi_upos)
            }
        except Exception as e:
            # Catch-all failsafe: If the model crashes for any internal reason (e.g., tensor size),
            # catch the error gracefully and return the default discard values instead of crashing the worker.
            return {
                "score": -1.0,
                "word_count": 0,
                "en_word_count": 0,
                "en_pos": "",
                "vi_pos": ""
            }
    
    scores, word_counts, en_word_counts = [], [], []
    en_pos_list, vi_pos_list, vi_text_list = [], [], []
    pos_blocks, text_blocks = [], []

    for en_text, vi_text in zip(examples['en'], examples['vi']):
        if not en_text or not vi_text:
            scores.append(-1.0)
            word_counts.append(0)
            en_word_counts.append(0)
            en_pos_list.append("")
            vi_pos_list.append("")
            vi_text_list.append("")
            pos_blocks.append("")
            text_blocks.append("")
            continue

        res = process_pair(en_text, vi_text)

        scores.append(res['score'])
        word_counts.append(res['word_count'])
        en_word_counts.append(res['en_word_count'])
        en_pos_list.append(res['en_pos'])
        vi_pos_list.append(res['vi_pos'])
        vi_text_list.append(vi_text)
        text_blocks.append(f"{en_text} \n\n {vi_text} [SPLIT]")
        pos_blocks.append(f"{res['en_pos']} \n\n {res['vi_pos']} [SPLIT]")

    return {
        "score": scores,
        "word_count": word_counts,
        "en_word_count": en_word_counts,
        "en_text": examples['en'],
        "en_pos": en_pos_list,
        "vi_text": vi_text_list,
        "vi_pos": vi_pos_list,
        "text_block": text_blocks,
        "pos_block": pos_blocks
    }

In [ ]:
from datasets import disable_progress_bar

disable_progress_bar()

print("Downloading from Hugging Face...")
# Load the entire train split. We will stop it dynamically.
dataset = load_dataset("ura-hcmut/PhoMT", split="train")

output_bilingual = "bilingual_training_data.jsonl"
output_english = "english_only_training_data.jsonl"

print("Some dataset samples:")
for i, item in enumerate(dataset):
	if i >= 5:  # Print only the first 5 samples
		break
	print(f"Sample {i + 1}: {item}")

In [ ]:
print("\nTẢI CÂU TỪ HUGGING FACE")
# Chỉ lấy đúng 10 câu đầu tiên cho nhanh
dataset = load_dataset("ura-hcmut/PhoMT", split="train[:2]")

print("\n=== BƯỚC 3: KẾT QUẢ ĐỐI CHIẾU EN - VI ===")
for i, item in enumerate(dataset):
    en_str = item.get('en', '').strip()
    vi_str = item.get('vi', '').strip()
    
    if not en_str or not vi_str:
        continue
        
    res = process_translation_pair(en_str, vi_str, nlp_en, nlp_vi)
    
    print(f"\n--- [CÂU SỐ {i+1}] ---")
    print(f"EN TEXT : {res['en_text']}")
    print(f"VI TEXT : {res['vi_text']}")
    print(f"EN POS  : {res['en_pos']}")
    print(f"VI POS  : {res['vi_pos']}")
    print(f"ĐỘ KHÓ  : {res['score']:.3f}")

In [ ]:
import multiprocess as mp

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

In [ ]:
full_dataset = load_dataset("ura-hcmut/PhoMT", split="train")
NUM_SHARDS = 3000 
output_dir = f'{HOME_DIR}/phomt_processed_shards'
os.makedirs(output_dir, exist_ok=True)
progress_bar = tqdm(total=NUM_SHARDS, desc="Processing 3M rows", unit="shard")

for i in range(NUM_SHARDS):
    shard_path = f"{output_dir}/shard_{i}"
    
    # Skip if already processed (allows resuming if the server crashes)
    if os.path.exists(shard_path):
        progress_bar.update(1)
        continue
        
    shard = full_dataset.shard(num_shards=NUM_SHARDS, index=i)
    
    processed_shard = shard.map(
        calculate_dataset_scores,
        batched=True,
        batch_size=1000, 
        num_proc=20,
    )
    
    processed_shard.save_to_disk(shard_path)
    
    progress_bar.update(1)
    
progress_bar.close()
print("Done, output: ", output_dir)

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import os

# Define the absolute path to your safe home directory
HOME_DIR = "/dss/dsshome1/0E/go54hem2"
shards_dir = f"{HOME_DIR}/phomt_processed_shards"
NUM_SHARDS = 3000

print("Loading shards...")
all_shards = []

# Loop through all expected shards and load them into memory
for i in range(NUM_SHARDS):
    shard_path = f"{shards_dir}/shard_{i}"
    
    if os.path.exists(shard_path):
        shard = load_from_disk(shard_path)
        all_shards.append(shard)
        print(f"Loaded shard {i+1}/{NUM_SHARDS}")
    else:
        # Warn if a shard is missing, but continue processing the rest
        print(f"Cannot find {shard_path}")

print(f"Loaded {len(all_shards)}/{NUM_SHARDS} shards.")
print("Concatenating data...")

# Concatenate all individual shard datasets into one single large dataset
final_dataset = concatenate_datasets(all_shards)

print(f"Done, total shards count: {len(final_dataset)}")

# Export the final dataset to a single Parquet file for efficient future loading
output_file = f"{HOME_DIR}/phomt_3M_scored_final.parquet"
print(f"Saving to: {output_file}")

final_dataset.to_parquet(output_file)

print("Done")

In [ ]:
file_path = f"{HOME_DIR}/phomt_3M_scored_final.parquet"
scored_dataset = load_dataset("parquet", data_files=file_path, split="train")

print("\n=== RESULTS ===")
for i in range(10):
    item = scored_dataset[i]
    print(f"\n[Sentence {i+1}]: {item['en_text']}")
    print(f"EN POS:  {item['en_pos']}")
    print(f"VI POS:  {item['vi_pos']}")
    print(f"SCORE:   {item['score']:.3f}")

In [ ]:
valid_dataset = scored_dataset.filter(lambda x: x["score"] != -1.0, num_proc=16)
print(valid_dataset)

In [ ]:
def update_score(example):
    en_upos = example["en_pos"].split()
    vi_upos = example["vi_pos"].split()

	base_length_score = len(en_upos) + len(vi_upos)

    en_sconj = en_upos.count("SCONJ") * 3.0
    en_cconj = en_upos.count("CCONJ") * 1.5
    en_punct = en_upos.count("PUNCT") * 0.5
    en_penalty = en_sconj + en_cconj + en_punct

	vi_sconj = vi_upos.count("SCONJ") * 3.0
    vi_cconj = vi_upos.count("CCONJ") * 1.5
    vi_punct = vi_upos.count("PUNCT") * 0.5
    vi_part = vi_upos.count("PART") * 1.0
    vi_penalty = vi_sconj + vi_cconj + vi_punct + vi_part

	difficulty_score = base_length_score + en_penalty + vi_penalty
    
    return {"score": difficulty_score}

new_scored_dataset = valid_dataset.map(update_score, num_proc=16)

new_output_file = f"{HOME_DIR}/phomt_3M_scored_new.parquet"
new_scored_dataset.to_parquet(new_output_file)
valid_dataset = load_dataset("parquet", data_files=new_output_file, split="train")

In [ ]:
HOME_DIR = "/dss/dsshome1/0E/go54hem2"

output_bilingual = f"{HOME_DIR}/bilingual_training_data.jsonl"
output_english = f"{HOME_DIR}/english_only_training_data.jsonl"
random.seed(42)

all_scores = valid_dataset["score"]
threshold_easy = np.percentile(all_scores, 45)
threshold_hard = np.percentile(all_scores, 80)

print("Distributing data into buckets...")
easy_bucket, medium_bucket, hard_bucket = [], [], []

for item in tqdm(valid_dataset, desc="Bucketing"):
	row_data = {
		"score": item["score"],
		"word_count": item["word_count"],
		"en_word_count": item["en_word_count"],
		"en_text": item["en"],
		"en_pos": item["en_pos"],
        "vi_text": item["vi"],
        "vi_pos": item["vi_pos"],
		"text_block": item["text_block"],
		"pos_block": item["pos_block"]
	}
	
	if row_data["score"] < threshold_easy:
		easy_bucket.append(row_data)
	elif row_data["score"] <= threshold_hard:
		medium_bucket.append(row_data)
	else:
		hard_bucket.append(row_data)

print(f"\nBucketing Summary: Easy={len(easy_bucket)}, Medium={len(medium_bucket)}, Hard={len(hard_bucket)}")

In [ ]:
import random
random.seed(42)

# Shuffle buckets to ensure semantic variety within the same difficulty level
random.shuffle(easy_bucket)
random.shuffle(medium_bucket)
random.shuffle(hard_bucket)

# Strict word budget constraint
STAGE1_TARGET = 1_000_000  # 10% Budget for pure POS framework
STAGE2_TARGET = 9_000_000  # 90% Budget for full lexical learning

print("\nExtracting Stage 1 dataset (1M words with balanced complexity)...")
stage1_data, s1_words = sample_from_buckets(easy_bucket, medium_bucket, hard_bucket, STAGE1_TARGET, stage_id=1)

print("Extracting Stage 2 dataset (9M words with balanced complexity)...")
stage2_data, s2_words = sample_from_buckets(easy_bucket, medium_bucket, hard_bucket, STAGE2_TARGET, stage_id=2)

print(f"Writing final structured file: {output_bilingual}")
with open(output_bilingual, "w", encoding="utf-8") as f_out:
	for item in stage1_data:
		f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
	for item in stage2_data:
		f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
		
print(f"Data engineering complete for bilingual data. Stage 1: {s1_words} words | Stage 2: {s2_words} words.")

print("\nExtracting English-only dataset (Target: 10M words)...")
ENG_TARGET = 10_000_000
english_dataset = []
eng_word_count = 0

# Reuse the English sentences from the bilingual set to guarantee overlap
bilingual_combined = stage1_data + stage2_data
for item in bilingual_combined:
	# Break immediately without adding if the next line exceeds target
	if eng_word_count + item["en_word_count"] > ENG_TARGET:
		break
	english_dataset.append({"text": item["en_text"], "pos": item["en_pos"]})
	eng_word_count += item["en_word_count"]
	
print(f"Overlap extracted: {eng_word_count} English words from the bilingual set.")

# Sample additional English sentences until we hit the 10M word target, using the same skewed curriculum approach
while eng_word_count < ENG_TARGET:
	if not easy_bucket and not medium_bucket and not hard_bucket:
		break
		
	pool = []
	weights = []
	if easy_bucket:
		pool.append('easy')
		weights.append(0.60)
	if medium_bucket:
		pool.append('medium')
		weights.append(0.30)
	if hard_bucket:
		pool.append('hard')
		weights.append(0.10)
		
	total_w = sum(weights)
	weights = [w / total_w for w in weights]
	
	chosen = random.choices(pool, weights=weights, k=1)[0]
	
	if chosen == 'easy': item = easy_bucket.pop()
	elif chosen == 'medium': item = medium_bucket.pop()
	else: item = hard_bucket.pop()
		
	# Break immediately without adding if the next line exceeds target
	if eng_word_count + item["en_word_count"] > ENG_TARGET:
		break
		
	english_dataset.append({"text": item["en_text"], "pos": item["en_pos"]})
	eng_word_count += item["en_word_count"]

print(f"Writing final English-only file: {output_english}")
with open(output_english, "w", encoding="utf-8") as f_out:
	for item in english_dataset:
		f_out.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Data engineering complete for English data. {eng_word_count} words")

In [ ]:
with open("/dss/dsshome1/0E/go54hem2/bilingual_training_data.jsonl", "r", encoding="utf-8") as f:
    print(repr(f.readline()))